# 99 — Final writeup

**Predicting customer churn on `playground-series-s6e3`** — a 594,194-row
synthetic expansion of the Telco churn table, scored by ROC-AUC.

**Result:** OOF **0.916567**, public LB **0.91401**, from a weighted blend of
LightGBM and CatBoost on 21 features. Folds frozen on day one and never
touched; every number below is reproducible from `experiments/`.

**How to read this notebook.** It trains nothing that gets scored — every score
comes from a saved `oof_*.npy` written by `src.train`, so nothing here can
quietly disagree with `experiments.csv`. The one model it does fit is a single
LightGBM fold, refitted purely to explain it, and cell 5.1 checks that refit is
*bit-identical* to the logged run before any explanation is built on it.

The three things worth your time, if you read nothing else:

1. **§3 — the noise floor.** What this setup can and cannot measure, derived
   rather than assumed. It is why most of the ideas below were rejected.
2. **§6 — the blend diagnostic.** The rule this repo started with turned out to
   be half a rule, and the missing half is an identity, not a heuristic.
3. **§7 — what the model says to *do*,** which is not the same as what it scores.

In [ ]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src import config as C
from src import cv as CV
from src.data import load
from src.utils import LOG_FILE, savefig

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)

log = pd.read_csv(LOG_FILE)
train, test = load()
y = train[C.TARGET].values

BEST = "0904_210905_blend"  # the submitted model
REF = "0904_202729_lgbm"  # best single model, and the reference for comparisons
oof = {r: np.load(C.EXPERIMENTS / f"oof_{r}.npy") for r in log.run_id}

print(f"{len(log)} logged runs, {len(oof)} saved OOF arrays")
print(f"base rate {y.mean():.4f}   train {train.shape}   test {test.shape}")

## 1. The problem, in business terms

A telco wants to know **which customers to spend retention budget on**. The
model outputs a churn probability per customer; the retention team works a
ranked call list from the top down until the budget runs out.

That operational shape is why **ROC-AUC is the right metric here and not
accuracy**. Accuracy asks "is the 0.5 threshold on the correct side?", which
nobody acts on. AUC asks "given one churner and one non-churner, does the model
put the churner higher?" — which is exactly the question a ranked call list
asks. At a 22.5% base rate, a model that predicts "nobody churns" is 77.5%
accurate and completely useless.

The number that makes this concrete for a business reader is **lift**: how much
richer in churners the top of the list is than the customer base as a whole.

In [ ]:
p = oof[BEST]
order = np.argsort(-p)
rows = []
for k in (0.01, 0.05, 0.10, 0.20, 0.30, 0.50):
    top = order[: int(k * len(p))]
    rows.append(
        {
            "call the top": f"{k:.0%}",
            "customers": len(top),
            "precision": y[top].mean(),
            "recall": y[top].sum() / y.sum(),
            "lift": y[top].mean() / y.mean(),
        }
    )
lift = pd.DataFrame(rows)
print(lift.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# (a) cumulative gains — the curve a retention manager actually reads
frac = np.arange(1, len(p) + 1) / len(p)
gains = np.cumsum(y[order]) / y.sum()
axes[0].plot(frac, gains, lw=2, label="model")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="random calling")
axes[0].set(
    xlabel="fraction of customers called",
    ylabel="fraction of churners reached",
    title="Cumulative gains (OOF)",
)
axes[0].legend()

# (b) how the ranking degrades as you go down the list
axes[1].plot(lift["call the top"], lift["lift"], "o-", lw=2)
axes[1].axhline(1.0, color="k", ls="--", lw=1, label="base rate")
axes[1].set(
    xlabel="depth of the call list", ylabel="lift over base rate", title="Lift by list depth"
)
axes[1].legend()

fig.tight_layout()
savefig(fig, "99_business_lift")

**Read:** the top 1% of the list is 96% churners — 4.3x the base rate — and the
top 10% still runs at 3.7x while reaching 37% of everyone who churns. By 30%
depth you have found 82% of all churners at 2.7x lift; the remaining 18% costs
another 20% of the base to reach, at a lift of 1.9x.

That shape is the useful part: **the budget question has a real answer, and the
answer is "go shallow"**. Past ~30% depth each additional call buys steeply less,
so extra budget is better spent deepening the *offer* to a short list than
lengthening the list.

Note what the lift table does **not** say: that calling these customers prevents
churn. The model ranks risk; it says nothing about whether an intervention
works. That needs an experiment, and §7 comes back to it.

## 2. The data, and what the EDA changed

Six findings from `notebooks/01_eda.ipynb`. The first three killed entire
families of features before any were built, which is the highest-value thing an
EDA does.

In [ ]:
facts = pd.DataFrame(
    [
        ("rows", f"train {len(train):,} / test {len(test):,} — test is exactly 30.0% of the total"),
        ("features", "19 raw: 4 numeric, 15 categorical at 2-4 levels each"),
        ("target", f"binary, {y.mean():.1%} positive"),
        (
            "missing cells",
            "zero in either frame — the classic Telco blank-totalcharges trap is absent",
        ),
        ("duplicate rows", f"{train.duplicated().sum()} in train"),
        ("time column", "none, and none derivable — id order carries no time signal"),
        ("entity column", "none — no customerid, and no feature vector repeats"),
        ("unseen levels", "none — every test category level occurs in train"),
        ("adversarial AUC", "0.5117 (train vs test on features alone) — same distribution"),
    ],
    columns=["fact", "value"],
)
print(facts.to_string(index=False))

**What each finding ruled out.**

| finding | family it killed |
|---|---|
| no time column | lags, rolling windows, any expanding-window CV |
| no repeated entity | group aggregates, customer history, `GroupKFold` |
| no missingness | imputation strategy, missing-indicator features |
| adversarial AUC 0.5117 | distribution-shift correction, sample reweighting, pseudo-labelling |
| no unseen levels | rare-level bucketing, unknown-category handling |

At ~849k rows the standard error on that adversarial AUC is ~0.0007, so 0.5117
is *measurably* above 0.5 and *practically* indistinguishable from it — the
distinction that matters, and the reason it reads as "no shift" rather than
"a small shift worth modelling".

What is left after those deletions is a wide, shallow, clean table with no
structure to exploit beyond the columns themselves. **That is the single fact
that determines everything downstream**, including why the ceiling turned out
to be so low and why a second model shape could not help (§6).

## 3. Validation design, and the noise floor

The test set is a random 30% slice of one table: one row per customer, no time,
no repeated entity, no shift. So **stratified 5-fold** mirrors it — stratified
rather than plain k-fold because at a 22.5% positive rate it removes
fold-to-fold class-balance wobble as a noise source for free.

Decided on day one and frozen. Every `oof_*.npy` carries a fingerprint of the
realised split, and `src/blend.py` refuses to average files whose fingerprints
disagree — because two OOF arrays made under different schemes are the same
shape and dtype, and averaging them yields a plausible number that means
nothing.

In [ ]:
fp = CV.fold_fingerprint(train)
folds = CV.get_folds(train)
print(f"scheme {C.CV_SCHEME}, {C.N_FOLDS} folds, fingerprint {fp}")
print(f"every logged run carries it: {set(log.folds.dropna()) == {fp}}\n")

print(
    pd.DataFrame(
        [
            {"fold": k, "n_train": len(tr), "n_valid": len(va), "pos_rate": y[va].mean()}
            for k, (tr, va) in enumerate(folds)
        ]
    ).to_string(index=False, float_format=lambda v: f"{v:.5f}")
)

# the folds partition the rows, so nothing is scored twice or never
covered = np.zeros(len(train), dtype=int)
for _, va in folds:
    covered[va] += 1
print(f"\nevery row validated exactly once: {bool((covered == 1).all())}")
print(f"no NaN in any saved OOF: {not any(np.isnan(v).any() for v in oof.values())}")

### What this setup can measure

Measured by bootstrap in `learnings.md`, not assumed:

| quantity | size |
|---|---|
| a single run's absolute fold score | ±0.00085 |
| a **paired** difference between two runs | ±0.000053 |
| the same config re-run under a different seed | ±0.000052 |

Runs share identical folds, so every comparison is **paired** — and that is
~16x more precise than either absolute score. The practical consequence is the
rule this project ran on: *anything smaller than the ~±0.00007 combined
paired-plus-seed noise is not a result, it is the seed.*

The cheap version, free on every run: **do the per-fold differences agree in
sign?** +0.0003 in 5/5 folds is real; +0.002 in 3/5 is one lucky fold.

In [ ]:
# The public LB is a *third* instrument, and the coarsest. Its resolution
# follows from the public set's size, which is not published — so bracket it.
scored = log.dropna(subset=["public_lb"]).copy()
scored["gap"] = scored.cv_score - scored.public_lb
print(scored[["run_id", "cv_score", "public_lb", "gap"]].to_string(index=False))

print("\npaired noise on an LB delta, by unknown public share:")
for share in (0.10, 0.20, 0.50):
    n = int(len(test) * share)
    # evaluation noise scales as 1/sqrt(n) from the known OOF paired-bootstrap SD
    sd = 0.000023 * np.sqrt(len(train) / n)
    print(f"  public = {share:>4.0%} of test ({n:>7,} rows):  SD ~ {sd:.6f}")
print("\nCV moves being made:  +0.000095 (features),  +0.000176 (blend)")

**Read — and this is the operational conclusion of the whole project.** At a
20% public set the paired noise on an LB delta is ~0.00008, while the changes
being made move the score 0.00010–0.00018. **The instrument is the same size as
the thing being measured.** One submission is worth roughly one bit, and that
bit is mostly noise.

The CV–LB gap is steady at −0.0024 to −0.0026 across all three scored runs. A
*constant* offset is what "CV tracks LB" looks like; it is the gap **moving**
that would signal the folds are wrong. It never moved.

So: **select on CV, and stop spending submissions to confirm changes this
small.** Both of those are decisions the noise floor made for us, not
preferences.

## 4. The experiment history

Nine runs. `experiments.csv` holds the numbers; `experiments.md` holds the
reasoning and the **Dead ends** table, which is the part that stops an idea
being tried twice.

In [ ]:
show = log.copy()
show["Δ vs baseline"] = show.cv_score - log.cv_score.iloc[0]
print(
    show[["run_id", "model", "cv_score", "Δ vs baseline", "public_lb"]].to_string(
        index=False,
        na_rep="",
        formatters={
            "cv_score": lambda v: f"{v:.6f}",
            "Δ vs baseline": lambda v: f"{v:+.6f}",
            "public_lb": lambda v: "" if pd.isna(v) else f"{v:.5f}",
        },
    )
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

# (a) every run's CV, in the order they were run
colors = {
    "lgbm": "tab:blue",
    "catboost": "tab:orange",
    "linear": "tab:green",
    "mlp": "tab:red",
    "blend": "tab:purple",
}
axes[0].bar(range(len(log)), log.cv_score, color=[colors[m] for m in log.model])
axes[0].set_ylim(0.905, 0.918)
axes[0].axhline(log.cv_score.iloc[0], color="k", ls="--", lw=1, label="baseline")
axes[0].set_xticks(range(len(log)))
axes[0].set_xticklabels([f"{r.split('_')[-1]}\n{i}" for i, r in enumerate(log.run_id)], fontsize=8)
axes[0].set(ylabel="OOF AUC", title="Every logged run")
axes[0].legend()

# (b) CV vs LB for the three scored runs
axes[1].plot(scored.cv_score, scored.public_lb, "o-", lw=2)
for _, r in scored.iterrows():
    axes[1].annotate(
        r.run_id.split("_")[-1],
        (r.cv_score, r.public_lb),
        textcoords="offset points",
        xytext=(6, -10),
        fontsize=8,
    )
axes[1].set(xlabel="OOF AUC", ylabel="public LB", title="CV vs public LB (3 scored runs)")

fig.tight_layout()
savefig(fig, "99_run_history")

**What actually moved the score, in order of size:**

| change | CV delta | verdict |
|---|---|---|
| blend of LightGBM + CatBoost | **+0.000176** | kept — 5/5 folds, 400/400 bootstrap |
| two charges ratios | **+0.000095** | kept — 5/5 folds, 393/400 bootstrap |
| `n_addons`, `tenure_bucket` | +0.000010 | **dropped** — a fifth of the seed spread |
| CatBoost alone | −0.000117 | kept as blend material only |
| MLP as a blend member | +0.00000000 | **dropped** (§6) |
| logistic regression as a blend member | +0.00000007 | **dropped** (§6) |

Two habits did most of the work here. **One change per run** — bundling four
features into one run is one measurement, not four, which is why the four
engineered features were immediately ablated and two of them thrown out. And
**the blend weight was checked for optimism** three ways, since `src.blend`
fits it on the same OOF it scores: fitted-on-all 0.916567, nested out-of-fold
0.916565, flat 50/50 with nothing fitted 0.916563. A spread of 0.000004 — one
weight on a 2-simplex fitted to 594k rows has nowhere to overfit.

## 5. What the model learned

### 5.1 Refitting the model we are about to explain

SHAP needs a fitted booster, and `src.train` saves predictions rather than
models. So one fold is refitted here — **not as a measurement**: it produces no
score, no OOF and no log row, and the repo's rule that every scored run goes
through `src.train` is intact.

What makes that legitimate is the assertion below. The refit must reproduce
`0904_202729_lgbm`'s fold-0 predictions **exactly**, bit for bit. If it does,
the explanation describes the model that was actually scored; if it did not,
everything in this section would be about some other model.

In [ ]:
from src.features import build_features, fold_features
from src.models.lgbm import LGBMModel
from src.utils import seed_everything

seed_everything()
tr_f, te_f, features = build_features(train, test)
tr_pos, va_pos = CV.get_folds(tr_f)[0]
X_tr, X_va, X_te, feats = fold_features(tr_f, te_f, tr_pos, va_pos, features)

fitted = LGBMModel().fit(X_tr, y[tr_pos], X_va, y[va_pos], features=feats)
refit = fitted.predict(X_va)
logged = oof[REF][va_pos]

assert np.array_equal(refit, logged), "refit differs from the logged run — do not explain it"
print(f"refit is bit-identical to {REF} on fold 0")
print(f"  trees {fitted.best_iteration}   fold-0 AUC {CV.score(y[va_pos], refit):.6f}")

In [ ]:
import shap

SAMPLE = 20_000
samp = X_va.iloc[:SAMPLE]
sv = np.asarray(shap.TreeExplainer(fitted.booster).shap_values(samp))

# cross-check against LightGBM's own TreeSHAP implementation
native = fitted.booster.predict(samp, num_iteration=fitted.best_iteration, pred_contrib=True)
print(f"shap matches lightgbm's native pred_contrib: {np.allclose(sv, native[:, :-1], atol=1e-6)}")
print(f"shap values {sv.shape}   base value (log-odds) {native[0, -1]:.4f}")

SHAP values are in **log-odds**, and they are additive: base value plus every
feature's contribution equals the model's output for that row. That additivity
is what makes the single-customer explanation in §5.4 an actual account of the
prediction rather than a plausible story about it.

The plots below are hand-rolled rather than `shap.summary_plot`, for one
reason: the features are pandas categoricals, and the stock plots want to
colour by numeric feature value. Same values, plots that read in the domain's
own language.

In [ ]:
mean_abs = pd.DataFrame({"feature": samp.columns, "mean_abs_shap": np.abs(sv).mean(0)}).sort_values(
    "mean_abs_shap", ascending=False
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.barplot(data=mean_abs.head(14), x="mean_abs_shap", y="feature", ax=axes[0], color="tab:blue")
axes[0].set(title="Global importance: mean |SHAP| (log-odds)", xlabel="mean |SHAP|")

# beeswarm-style: spread of signed contributions for the top features
top_f = mean_abs.head(10).feature.tolist()
long = pd.DataFrame(sv, columns=samp.columns)[top_f].melt(var_name="feature", value_name="shap")
sns.stripplot(
    data=long.sample(6000, random_state=C.SEED),
    x="shap",
    y="feature",
    ax=axes[1],
    size=2,
    alpha=0.25,
    color="tab:blue",
)
axes[1].axvline(0, color="k", lw=1)
axes[1].set(title="Signed contribution, spread across customers", xlabel="SHAP (log-odds)")

fig.tight_layout()
savefig(fig, "99_shap_global")
print(mean_abs.head(10).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

### 5.2 The drivers, in the domain's language

`contract` dominates, and by a wide margin. That is not a modelling artefact —
it is the single most actionable fact the model produces, because contract type
is the one driver on this list the business **directly controls**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
sv_df = pd.DataFrame(sv, columns=samp.columns)

# (a) contract — the dominant driver, by level
d = pd.DataFrame({"contract": samp.contract.astype(str).values, "shap": sv_df.contract.values})
sns.boxplot(data=d, x="shap", y="contract", ax=axes[0], fliersize=1)
axes[0].axvline(0, color="k", lw=1)
axes[0].set(title="contract: contribution by level", xlabel="SHAP (log-odds)")

# (b) tenure — the strongest numeric, and clearly non-linear
axes[1].scatter(samp.tenure, sv_df.tenure, s=3, alpha=0.15)
axes[1].axhline(0, color="k", lw=1)
axes[1].set(xlabel="tenure (months)", ylabel="SHAP", title="tenure: contribution vs value")

# (c) does the engineered ratio earn its place?
axes[2].scatter(samp.charges_ratio, sv_df.charges_ratio, s=3, alpha=0.15, color="tab:green")
axes[2].axhline(0, color="k", lw=1)
axes[2].set(xlabel="charges_ratio", ylabel="SHAP", title="charges_ratio: the engineered feature")

fig.tight_layout()
savefig(fig, "99_shap_drivers")

In [ ]:
# the same thing as a table a non-technical reader can act on
drivers = []
for col in ["contract", "paymentmethod", "internetservice", "onlinesecurity"]:
    g = pd.DataFrame({"level": samp[col].astype(str).values, "shap": sv_df[col].values})
    for level, sub in g.groupby("level"):
        drivers.append(
            {
                "feature": col,
                "level": level,
                "n": len(sub),
                "mean_shap": sub.shap.mean(),
                "direction": "raises risk" if sub.shap.mean() > 0 else "lowers risk",
            }
        )
print(pd.DataFrame(drivers).to_string(index=False, float_format=lambda v: f"{v:+.4f}"))

**What the model learned, stated plainly:**

- **Month-to-month contracts raise churn risk more than anything else in the
  data** (mean SHAP +1.51 log-odds), and two-year contracts lower it just as
  hard (−1.81). The lever this implies is a contract migration offer, not a
  discount.
- **Electronic-check payment raises risk** (+0.56) while every automatic method
  lowers it. Payment friction is a churn signal, and it is cheap to change.
- **Fibre-optic internet raises risk** (+0.46) relative to DSL — a
  price-and-expectation effect rather than a technical one.
- **`tenure` is strongly non-linear**: it contributes ~+1.5 log-odds at month
  zero, falls steeply over the first year and a half, crosses into
  risk-*lowering* around month 17-18, then keeps drifting down more gently to
  about −0.5 by year six. A new customer and a six-year customer are close to
  opposite signals from the same column — which is why a purely additive model
  in a one-hot basis cannot represent this well, and part of why the linear
  model in §6 trailed by 0.0084.

**On the engineered features, read the table honestly.** `avg_monthly_spend`
ranks 6th by mean |SHAP| while `charges_ratio` sits outside the top ten. Its
dependence plot is clean — a customer billed more than their current rate
implies is monotonically riskier — but the *magnitude* is small, so SHAP alone
would not justify keeping the column. What justifies it is the ablation in
`experiments.md`: the two ratios *together* are worth +0.000095 over the
baseline, positive in 5/5 folds, while an add-on count and a tenure bucket added
+0.000010 and were dropped. **Importance ranks contribution to this model; the
ablation measures what a feature is worth.** They are different questions, and
only the second one licenses keeping a column.

The same distinction explains `tenure`: it is the strongest *single* feature in
notebook 01's univariate ranking, yet only 4th here, because `contract` already
encodes much of what it carried.

### 5.3 Explaining one customer

Additivity means a single prediction decomposes exactly. This is the artefact a
retention agent would actually see next to a name on the call list.

In [ ]:
i = int(np.argsort(-refit[:SAMPLE])[0])  # the highest-risk customer in the sample
contrib = (
    pd.DataFrame({"feature": samp.columns, "value": samp.iloc[i].astype(str).values, "shap": sv[i]})
    .sort_values("shap", key=np.abs, ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["tab:red" if v > 0 else "tab:blue" for v in contrib.shap]
ax.barh([f"{f} = {v}" for f, v in zip(contrib.feature, contrib.value)], contrib.shap, color=colors)
ax.axvline(0, color="k", lw=1)
ax.invert_yaxis()
ax.set(
    xlabel="SHAP (log-odds)  —  red raises risk, blue lowers it",
    title=f"Why customer #{i} is ranked highest (P(churn) = {refit[i]:.3f})",
)
fig.tight_layout()
savefig(fig, "99_shap_one_customer")

base_v = native[i, -1]
print(
    f"base log-odds {base_v:+.4f}  + contributions {sv[i].sum():+.4f}  = {base_v + sv[i].sum():+.4f}"
)
print(f"sigmoid -> {1 / (1 + np.exp(-(base_v + sv[i].sum()))):.4f}   model output {refit[i]:.4f}")

## 6. Choosing a blend partner — where the received wisdom broke

This is the part of the project worth reading if you are going to reuse
anything from it.

The `README` ranks "a blend of *structurally different* models" third by value
per hour, and warns that two boosting libraries on the same features correlate
~0.9 and blend to almost nothing. Both halves of that turned out to be wrong
here, in opposite directions — and fixing them produced a diagnostic that is
now the repo's rule for what to blend.

**First break: the correlation was 0.997, and the blend worked anyway.**
LightGBM and CatBoost correlated far *higher* than the README predicted, which
made "worth nothing" look safe. The blend gained +0.000176 — twice what all the
feature engineering bought. The correlation is a Pearson coefficient of two
prediction vectors, dominated by the 54% of rows where both models say "will
not churn" and agree trivially. **The metric is a ranking statistic, so the
diagnostic has to be one too.**

In [ ]:
def dissent(a, b, n_pairs=2_000_000):
    """Decompose two runs' ranking disagreement: how often they order a
    (churner, non-churner) pair differently, and who is right when they do."""
    rng = np.random.default_rng(C.SEED)
    i = rng.choice(np.flatnonzero(y == 1), n_pairs)
    j = rng.choice(np.flatnonzero(y == 0), n_pairs)
    ok_a, ok_b = oof[a][i] > oof[a][j], oof[b][i] > oof[b][j]
    a_only, b_only = (ok_a & ~ok_b).mean(), (~ok_a & ok_b).mean()
    d = a_only + b_only
    return {"disagree": d, "a_right": a_only / d, "gap": CV.score(y, oof[a]) - CV.score(y, oof[b])}


CANDIDATES = ["0904_210834_catboost", "0905_020138_mlp", "0905_015118_linear"]
rows = []
for c in CANDIDATES:
    r = dissent(REF, c)
    rows.append(
        {
            "candidate": c.split("_")[-1],
            "solo AUC": CV.score(y, oof[c]),
            "gap": r["gap"],
            "correlation": np.corrcoef(oof[REF], oof[c])[0, 1],
            "disagree %": 100 * r["disagree"],
            "lgbm right %": 100 * r["a_right"],
            "identity says %": 100 * (0.5 + r["gap"] / (2 * r["disagree"])),
        }
    )
diag = pd.DataFrame(rows)
print(
    diag.to_string(
        index=False,
        formatters={
            "solo AUC": lambda v: f"{v:.6f}",
            "gap": lambda v: f"{v:+.6f}",
            "correlation": lambda v: f"{v:.3f}",
            "disagree %": lambda v: f"{v:.2f}",
            "lgbm right %": lambda v: f"{v:.1f}",
            "identity says %": lambda v: f"{v:.1f}",
        },
    )
)

**Second break: ranking disagreement, read on its own, picks the *worst*
candidate.**

Look at the `disagree %` column against what each candidate is actually worth.
The logistic regression disagrees with LightGBM on 3.65% of pairs — more than
three times CatBoost's 1.10% — and by the "disagreement = blend material" rule
it should be the best partner available. It is worth nothing. So is the MLP.

The missing half of the diagnostic is **who is right when they disagree**, and
that column settles it: CatBoost's dissent is a coin flip (50.4%), the others
are predictably wrong (57.3%, 61.5%). Averaging can recover information from a
disagreement that is a coin flip. It can recover nothing from a disagreement
that is reliably in one direction.

**And the balance is not an observation — it is forced.** AUC *is* the
probability of ordering a (churner, non-churner) pair correctly, so for any two
models

$$\text{AUC}_A - \text{AUC}_B = P(A\text{ right}, B\text{ wrong}) - P(B\text{ right}, A\text{ wrong})$$

Both terms sum to the disagreement $d$, so the balance follows immediately:

$$P(A \text{ right} \mid \text{they disagree}) = \frac{1}{2} + \frac{\text{AUC}_A - \text{AUC}_B}{2d}$$

A model's solo deficit **is** its net loss on the pairs where it dissents;
there is nowhere else for a deficit to live. The `identity says %` column above
is that formula, and it matches the measured column to within 0.3pp on all
three models.

In [ ]:
# Does any weight on the candidate improve on LightGBM alone? Same base for all
# three, so the comparison is symmetric.
ws = np.arange(0, 0.61, 0.02)
curves = {
    c: np.array([CV.score(y, (1 - w) * oof[REF] + w * oof[c]) for w in ws]) for c in CANDIDATES
}
base = CV.score(y, oof[REF])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

for c in CANDIDATES:
    axes[0].plot(ws, curves[c] - base, lw=2, label=c.split("_")[-1])
axes[0].axhline(0, color="k", ls="--", lw=1)
axes[0].set(
    xlabel="weight on the candidate",
    ylabel="AUC gain over LightGBM alone",
    title="Only one candidate is worth any weight",
)
axes[0].legend()

# the identity, checked
axes[1].plot([49, 63], [49, 63], "k--", lw=1, label="perfect agreement")
axes[1].scatter(diag["identity says %"], diag["lgbm right %"], s=90, zorder=3)
for _, r in diag.iterrows():
    axes[1].annotate(
        r.candidate,
        (r["identity says %"], r["lgbm right %"]),
        textcoords="offset points",
        xytext=(8, -4),
        fontsize=9,
    )
axes[1].set(
    xlabel="predicted by 1/2 + gap/2d",
    ylabel="measured",
    title="Balance of dissent: identity vs measurement",
)
axes[1].legend()

fig.tight_layout()
savefig(fig, "99_blend_diagnostic")

print("best achievable gain over LightGBM alone:")
for c in CANDIDATES:
    k = int(curves[c].argmax())
    print(f"  {c.split('_')[-1]:>9}: {curves[c][k] - base:+.8f} at w={ws[k]:.2f}")

In [ ]:
# Inverting the identity gives a hiring bar: what solo score would a candidate
# need for its dissent to be as even as CatBoost's? CatBoost itself is excluded —
# it is what the bar is calibrated to, so scoring it against itself is circular.
lgbm_auc = CV.score(y, oof[REF])
print(f"reference: {REF} at {lgbm_auc:.6f}\n")
print("solo AUC needed for balanced dissent, at each candidate's own disagreement rate:\n")
for _, r in diag[diag.candidate != "catboost"].iterrows():
    need = lgbm_auc - 0.01 * (r["disagree %"] / 100)
    print(
        f"  {r.candidate:>7}: d={r['disagree %']:.2f}%  needs >= {need:.6f}  "
        f"has {r['solo AUC']:.6f}  short by {need - r['solo AUC']:.6f}  "
        f"({(need - r['solo AUC']) / 0.00024:.0f}x the bar's own width)"
    )

**The bar, and the verdict on structural diversity.**

Inverted, the identity says a candidate needs `gap ≤ 0.01 × d` to dissent as
evenly as CatBoost does. At the disagreement rates actually observed, that puts
the bar **within ~0.00024 of LightGBM** — which on this dataset means *being as
good as a tuned booster*. The MLP is 0.0034 short and the linear model 0.0081,
i.e. 14x and 34x further away than the bar allows.

So a blend member must be **different *and* competitive, and different is the
cheap half.** The only pair that ever qualified here is the two boosters sitting
0.000117 apart — precisely the pair the README predicted would be worth almost
nothing.

Two honest limits on that conclusion. The MLP is untuned: its patience was
raised on the argument that sklearn early-stops on *accuracy*, a poor proxy for
AUC at a 22.5% base rate, but it was never tuned against a fold — doing so would
have meant selecting on the rows the OOF is computed from. And the verdict is
dataset-specific: §2 established that this is a clean, shallow, shift-free table
where nothing that is not a tree comes close. **What generalises is the test,
not the verdict** — and the test costs nothing, since the gap is two numbers
already in `experiments.csv`, and `src.blend` prints the disagreement matrix,
the who-is-right split, and `1/2 + gap/2d` beside the measurement on every run.

## 7. Recommendation, and what this does not license

### What to do with the model

1. **Work the top decile, not the top half.** §1 shows the gains curve is steep
   early and flat after ~30%. Retention budget is better spent deepening the
   *offer* to a shallow list than lengthening the list.
2. **The strongest lever is contract type, not price.** `contract` dominates
   every importance measure computed here, and unlike `tenure` or
   `internetservice` it is something the business can offer to change. A
   month-to-month customer moved to an annual contract moves along the single
   most important axis the model has.
3. **Fix payment friction.** Electronic-check payment raises risk while
   automatic methods lower it. That is among the cheapest interventions
   available and needs no discount.

### What this does not license

- **The model ranks risk; it does not identify persuadable customers.** A
  customer who will churn no matter what and one who can be retained look
  identical to it. Acting on it optimally needs an uplift model, and that needs
  an experiment — a randomised holdout on the call list — which no observational
  fit can substitute for.
- **Nothing here is causal.** "Month-to-month raises churn risk" is a statement
  about the data, not a promise that migrating a customer to an annual contract
  lowers their risk. Contract type is partly a *consequence* of intent to stay.
- **The data is synthetic**, a generated expansion of the Telco table. Feature
  relationships are inherited from a generator, not from a real customer base,
  so the domain conclusions above are demonstrations of method rather than
  advice to a real operator.
- **The score is at its ceiling and the leaderboard cannot resolve what is
  left.** Five models span 0.0086 while the top two sit 0.000117 apart; §3
  showed the LB's paired noise (~0.00008) is the size of the remaining moves.

### What I would do next, and what I would not

**Would not:** add more model families. §6 measured the bar and nothing
available clears it. **Would not:** spend submissions to confirm changes below
~0.0002 — §3 shows they carry about one bit each, mostly noise.

**Would**, in order: (1) hyperparameter tuning on LightGBM with `optuna`, the
one untried lever and the one the README ranks last, for an expected
+0.0002–0.0005; (2) if a real deployment were the goal, an uplift experiment,
which is worth more than every AUC decimal in this notebook put together.

---

### Reproducing this

```bash
make setup                                   # uv sync + pre-commit hooks
make data                                    # download to data/raw/ (gitignored)
uv run python -m src.train --model lgbm --notes "baseline"
uv run python -m src.blend 0904_202729_lgbm 0904_210834_catboost
```

Every figure in this notebook is written to `reports/figures/` as it is made.
`experiments.csv` holds the numbers, `experiments.md` the reasoning and the
**Dead ends** table, and `learnings.md` the derivations behind §3's noise floor.
Folds are frozen at fingerprint `13feb807`; `src/blend.py` refuses to average
OOF files that disagree with it.